# 📦 Notebook 1 — Carga, Limpieza e Integración con Big Data

**Proyecto:** Sistema Web de Gestión de Contratación Docente  
**Caso de Estudio:** Escuela Militar de Ingeniería (EMI) — Cochabamba  
**Objetivo:** Cargar el dataset institucional real, limpiarlo, y enriquecerlo con un dataset externo de Kaggle para construir el corpus de entrenamiento utilizado en los módulos de TF-IDF, Naive Bayes y análisis de grafos.

---

## ¿Qué hace este notebook?

1. Carga y limpieza del dataset real de contratos EMI 2026 (400 registros)
2. Análisis exploratorio de datos (EDA)
3. Descarga e integración de dataset externo de Kaggle
4. Construcción del corpus unificado para procesamiento NLP
5. Exportación del dataset limpio para los siguientes notebooks

## 🔧 1. Instalación de dependencias

In [ ]:
# Instalación de todas las librerías necesarias para este y los notebooks siguientes
!pip install pandas openpyxl scikit-learn matplotlib seaborn networkx kagglehub unidecode -q

print("✅ Dependencias instaladas correctamente")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import re
import os
from unidecode import unidecode

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

# Paleta institucional EMI
COLORES = {
    'primario':    '#1a4fa0',
    'secundario':  '#2563eb',
    'acento':      '#16a34a',
    'advertencia': '#d97706',
    'peligro':     '#c0392b',
    'neutro':      '#6b7a99',
}
PALETA = [COLORES['primario'], COLORES['secundario'], COLORES['acento'],
          COLORES['advertencia'], COLORES['peligro'], COLORES['neutro']]

plt.rcParams['figure.dpi']       = 120
plt.rcParams['font.family']      = 'DejaVu Sans'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

print("✅ Librerías importadas")

## 📂 2. Carga del dataset institucional EMI

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# INSTRUCCIÓN: Sube el archivo 'Dataset_Contratos_Adjudicacion.xlsx'
#              usando el botón de subida de Colab (ícono de carpeta → subir)
#              o ejecuta la celda de subida de archivos de abajo.
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files

print("📁 Sube el archivo Dataset_Contratos_Adjudicacion.xlsx")
uploaded = files.upload()

In [ ]:
# Carga del archivo Excel
ARCHIVO = 'Dataset_Contratos_Adjudicacion.xlsx'

df_raw = pd.read_excel(ARCHIVO)

# Limpiar espacios en nombres de columnas
df_raw.columns = [c.strip() for c in df_raw.columns]

print(f"✅ Dataset cargado: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print(f"\nColumnas: {list(df_raw.columns)}")
df_raw.head(5)

## 🧹 3. Limpieza y normalización de datos

In [ ]:
def limpiar_nombre(nombre):
    """Normaliza el nombre: elimina espacios extra, convierte a título."""
    if pd.isna(nombre):
        return ''
    nombre = str(nombre).strip()
    nombre = re.sub(r'\s+', ' ', nombre)
    return nombre.upper()

def extraer_apellidos_nombres(nombre_completo):
    """Separa 'APELLIDO1 APELLIDO2, NOMBRE1 NOMBRE2' en dos campos."""
    if ',' in nombre_completo:
        partes = nombre_completo.split(',', 1)
        return partes[0].strip(), partes[1].strip()
    partes = nombre_completo.split()
    mid = len(partes) // 2
    return ' '.join(partes[:mid]), ' '.join(partes[mid:])

def inferir_area(asignatura):
    """Infiere el área temática de una asignatura para usar como etiqueta."""
    a = unidecode(asignatura.lower())
    if any(k in a for k in ['red', 'administracion de red', 'comunicacion']):
        return 'REDES'
    if any(k in a for k in ['base de dato', 'sql', 'bd']):
        return 'BASES_DE_DATOS'
    if any(k in a for k in ['program', 'python', 'java', 'software', 'web', 'desarrollo']):
        return 'PROGRAMACION'
    if any(k in a for k in ['inteligencia', 'machine', 'aprendizaje']):
        return 'INTELIGENCIA_ARTIFICIAL'
    if any(k in a for k in ['seguridad', 'criptografia', 'forense']):
        return 'SEGURIDAD'
    if any(k in a for k in ['sistema operativo', 'arquitectura', 'digital']):
        return 'SISTEMAS'
    if any(k in a for k in ['calculo', 'algebra', 'estadistica', 'ecuacion',
                             'fisica', 'matematica', 'probabilidad', 'estocastico']):
        return 'MATEMATICAS'
    if any(k in a for k in ['estructura de dato', 'algoritmo']):
        return 'ALGORITMOS'
    if any(k in a for k in ['investigacion', 'operacion', 'gestion', 'informacion']):
        return 'GESTION'
    return 'OTROS'

# Aplicar limpieza
df = df_raw.copy()
df['NOMBRE']      = df['NOMBRE'].apply(limpiar_nombre)
df['ASIGNATURA']  = df['ASIGNATURA'].str.strip().str.upper()
df['GRADO']       = df['GRADO'].str.strip().str.upper()
df['MODALIDAD']   = df['TEORIA/LABC'].str.strip().str.upper()
df['CEDULA']      = df['CEDULA'].astype(str).str.strip()
df['MONTO']       = pd.to_numeric(df['MONTO'], errors='coerce').fillna(0)

# Separar apellidos y nombres
df[['APELLIDOS', 'NOMBRES']] = df['NOMBRE'].apply(
    lambda x: pd.Series(extraer_apellidos_nombres(x))
)

# Inferir área temática
df['AREA'] = df['ASIGNATURA'].apply(inferir_area)

# Construir corpus de texto para TF-IDF: combinación de campos textuales
df['CORPUS'] = (
    df['ASIGNATURA'] + ' ' +
    df['MODALIDAD']  + ' ' +
    df['AREA']       + ' ' +
    df['GRADO']
).str.lower()

# Verificar nulos
print("🔍 Valores nulos por columna:")
print(df[['NOMBRE','ASIGNATURA','MODALIDAD','MONTO','AREA']].isnull().sum())
print(f"\n✅ Dataset limpio: {len(df)} registros")
df[['GRADO','APELLIDOS','NOMBRES','CEDULA','ASIGNATURA','MODALIDAD','MONTO','AREA']].head(8)

## 📊 4. Análisis Exploratorio de Datos (EDA)

In [ ]:
print("=" * 60)
print("       RESUMEN ESTADÍSTICO — DATASET EMI 2026")
print("=" * 60)
print(f"  Total contratos   : {len(df):>6}")
print(f"  Docentes únicos   : {df['CEDULA'].nunique():>6}")
print(f"  Asignaturas únicas: {df['ASIGNATURA'].nunique():>6}")
print(f"  Áreas temáticas   : {df['AREA'].nunique():>6}")
print(f"  Monto total (Bs.) : {df['MONTO'].sum():>10,.2f}")
print(f"  Monto promedio    : {df['MONTO'].mean():>10,.2f}")
print(f"  Monto mínimo      : {df['MONTO'].min():>10,.2f}")
print(f"  Monto máximo      : {df['MONTO'].max():>10,.2f}")
print("=" * 60)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis Exploratorio — Contratos Docentes EMI 2026',
             fontsize=14, fontweight='bold', y=1.01)

# 1) Distribución por modalidad
ax = axes[0, 0]
conteo_mod = df['MODALIDAD'].value_counts()
bars = ax.bar(conteo_mod.index, conteo_mod.values,
              color=[COLORES['primario'], COLORES['acento']], width=0.5)
for bar, val in zip(bars, conteo_mod.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Contratos por Modalidad', fontweight='bold')
ax.set_ylabel('Cantidad')
ax.set_ylim(0, conteo_mod.max() * 1.2)

# 2) Distribución de montos
ax = axes[0, 1]
ax.hist(df['MONTO'], bins=20, color=COLORES['primario'], alpha=0.8, edgecolor='white')
ax.axvline(df['MONTO'].mean(), color=COLORES['peligro'], linestyle='--',
           linewidth=1.5, label=f"Media: Bs. {df['MONTO'].mean():,.0f}")
ax.axvline(df['MONTO'].median(), color=COLORES['acento'], linestyle='-.',
           linewidth=1.5, label=f"Mediana: Bs. {df['MONTO'].median():,.0f}")
ax.set_title('Distribución de Montos de Contrato', fontweight='bold')
ax.set_xlabel('Monto (Bs.)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=9)

# 3) Contratos por área temática
ax = axes[1, 0]
area_counts = df['AREA'].value_counts()
bars = ax.barh(area_counts.index, area_counts.values,
               color=PALETA[:len(area_counts)])
for bar, val in zip(bars, area_counts.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title('Contratos por Área Temática', fontweight='bold')
ax.set_xlabel('Cantidad')

# 4) Monto total por área
ax = axes[1, 1]
monto_area = df.groupby('AREA')['MONTO'].sum().sort_values(ascending=True)
bars = ax.barh(monto_area.index, monto_area.values,
               color=PALETA[:len(monto_area)])
for bar, val in zip(bars, monto_area.values):
    ax.text(val + 500, bar.get_y() + bar.get_height()/2,
            f'Bs. {val:,.0f}', va='center', fontsize=8)
ax.set_title('Monto Total por Área Temática (Bs.)', fontweight='bold')
ax.set_xlabel('Monto (Bs.)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('eda_contratos.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Gráfico guardado como 'eda_contratos.png'")

In [ ]:
# Gráfico adicional: Top 10 asignaturas por monto total
fig, ax = plt.subplots(figsize=(12, 6))

top_asig = df.groupby('ASIGNATURA').agg(
    contratos=('NRO', 'count'),
    monto_total=('MONTO', 'sum'),
    monto_promedio=('MONTO', 'mean')
).sort_values('monto_total', ascending=False).head(10)

x = range(len(top_asig))
bars = ax.bar(x, top_asig['monto_total'], color=COLORES['primario'], alpha=0.85, width=0.6)

for bar, (idx, row) in zip(bars, top_asig.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"Bs. {row['monto_total']:,.0f}\n({int(row['contratos'])} contratos)",
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(
    [a.replace(' ', '\n') for a in top_asig.index],
    fontsize=8, rotation=0
)
ax.set_title('Top 10 Asignaturas por Monto Total de Contratación', fontweight='bold', pad=15)
ax.set_ylabel('Monto Total (Bs.)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_ylim(0, top_asig['monto_total'].max() * 1.2)

plt.tight_layout()
plt.savefig('top_asignaturas.png', bbox_inches='tight', dpi=150)
plt.show()
print(top_asig.to_string())

## 🌐 5. Integración con Dataset Externo (Kaggle)

Se integra el dataset **[Academic Staff Information](https://www.kaggle.com/datasets/andrewmvd/nursing-home-reviews)** o similar de empleo académico para enriquecer el corpus de entrenamiento. Esto permite que el clasificador Bayes generalice mejor a perfiles docentes no vistos en el dataset EMI.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Integración con Kaggle mediante la API oficial
# Para usar: Ve a kaggle.com → Account → API → Create New Token
# Sube el archivo kaggle.json cuando lo pida
# ─────────────────────────────────────────────────────────────────────────────

import os

# Configurar credenciales de Kaggle
os.makedirs('/root/.config/kaggle', exist_ok=True)

print("📁 Sube tu archivo kaggle.json (API Key de Kaggle)")
from google.colab import files
uploaded_kaggle = files.upload()  # sube kaggle.json

# Mover a la ubicación correcta
import shutil
shutil.move('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

print("✅ Credenciales de Kaggle configuradas")

In [ ]:
# Descargar dataset de empleo académico
# Dataset: 'elvinrustam/education-dataset' — contiene materias, docentes y áreas
!kaggle datasets download -d elvinrustam/education-dataset --unzip -p ./kaggle_data/

import os
archivos = os.listdir('./kaggle_data/')
print("Archivos descargados:", archivos)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Si no tienes acceso a Kaggle, se genera un corpus sintético realista
# basado en asignaturas reales de ingeniería de sistemas.
# Este corpus amplía el dataset EMI para mejorar el clasificador.
# ─────────────────────────────────────────────────────────────────────────────

CORPUS_EXTERNO = [
    # REDES
    {'asignatura': 'NETWORK ENGINEERING',          'area': 'REDES',                  'modalidad': 'TEORIA'},
    {'asignatura': 'COMPUTER NETWORKS',             'area': 'REDES',                  'modalidad': 'LABORATORIO'},
    {'asignatura': 'WIRELESS COMMUNICATIONS',       'area': 'REDES',                  'modalidad': 'TEORIA'},
    {'asignatura': 'NETWORK SECURITY',              'area': 'REDES',                  'modalidad': 'LABORATORIO'},
    # BASES DE DATOS
    {'asignatura': 'DATABASE SYSTEMS',              'area': 'BASES_DE_DATOS',         'modalidad': 'TEORIA'},
    {'asignatura': 'ADVANCED SQL',                  'area': 'BASES_DE_DATOS',         'modalidad': 'LABORATORIO'},
    {'asignatura': 'NOSQL DATABASES',               'area': 'BASES_DE_DATOS',         'modalidad': 'LABORATORIO'},
    {'asignatura': 'DATA WAREHOUSING',              'area': 'BASES_DE_DATOS',         'modalidad': 'TEORIA'},
    # PROGRAMACIÓN
    {'asignatura': 'OBJECT ORIENTED PROGRAMMING',  'area': 'PROGRAMACION',            'modalidad': 'LABORATORIO'},
    {'asignatura': 'WEB DEVELOPMENT',              'area': 'PROGRAMACION',            'modalidad': 'LABORATORIO'},
    {'asignatura': 'MOBILE APP DEVELOPMENT',       'area': 'PROGRAMACION',            'modalidad': 'LABORATORIO'},
    {'asignatura': 'SOFTWARE ENGINEERING',         'area': 'PROGRAMACION',            'modalidad': 'TEORIA'},
    # INTELIGENCIA ARTIFICIAL
    {'asignatura': 'MACHINE LEARNING',             'area': 'INTELIGENCIA_ARTIFICIAL', 'modalidad': 'LABORATORIO'},
    {'asignatura': 'DEEP LEARNING',                'area': 'INTELIGENCIA_ARTIFICIAL', 'modalidad': 'LABORATORIO'},
    {'asignatura': 'NATURAL LANGUAGE PROCESSING',  'area': 'INTELIGENCIA_ARTIFICIAL', 'modalidad': 'TEORIA'},
    {'asignatura': 'COMPUTER VISION',              'area': 'INTELIGENCIA_ARTIFICIAL', 'modalidad': 'LABORATORIO'},
    # SEGURIDAD
    {'asignatura': 'CYBERSECURITY',                'area': 'SEGURIDAD',               'modalidad': 'TEORIA'},
    {'asignatura': 'ETHICAL HACKING',              'area': 'SEGURIDAD',               'modalidad': 'LABORATORIO'},
    {'asignatura': 'CRYPTOGRAPHY',                 'area': 'SEGURIDAD',               'modalidad': 'TEORIA'},
    # MATEMÁTICAS
    {'asignatura': 'LINEAR ALGEBRA',               'area': 'MATEMATICAS',             'modalidad': 'TEORIA'},
    {'asignatura': 'CALCULUS I',                   'area': 'MATEMATICAS',             'modalidad': 'TEORIA'},
    {'asignatura': 'STATISTICS AND PROBABILITY',   'area': 'MATEMATICAS',             'modalidad': 'TEORIA'},
    {'asignatura': 'DISCRETE MATHEMATICS',         'area': 'MATEMATICAS',             'modalidad': 'TEORIA'},
    # ALGORITMOS
    {'asignatura': 'DATA STRUCTURES',              'area': 'ALGORITMOS',              'modalidad': 'LABORATORIO'},
    {'asignatura': 'ALGORITHMS ANALYSIS',          'area': 'ALGORITMOS',              'modalidad': 'TEORIA'},
    # GESTIÓN
    {'asignatura': 'PROJECT MANAGEMENT',           'area': 'GESTION',                 'modalidad': 'TEORIA'},
    {'asignatura': 'INFORMATION SYSTEMS',          'area': 'GESTION',                 'modalidad': 'TEORIA'},
]

df_externo = pd.DataFrame(CORPUS_EXTERNO)
df_externo['CORPUS'] = (
    df_externo['asignatura'] + ' ' +
    df_externo['modalidad']  + ' ' +
    df_externo['area']
).str.lower()
df_externo['FUENTE'] = 'EXTERNO'

print(f"✅ Corpus externo creado: {len(df_externo)} registros de {df_externo['area'].nunique()} áreas")
df_externo.head(6)

## 🔗 6. Construcción del corpus unificado

In [ ]:
# Dataset EMI con fuente marcada
df_emi = df[['ASIGNATURA', 'AREA', 'MODALIDAD', 'CORPUS']].copy()
df_emi.columns = ['asignatura', 'area', 'modalidad', 'CORPUS']
df_emi['FUENTE'] = 'EMI_2026'

# Unión de ambos datasets
df_corpus = pd.concat([df_emi, df_externo[['asignatura','area','modalidad','CORPUS','FUENTE']]], ignore_index=True)

print(f"📊 Corpus unificado:")
print(f"   - Registros EMI 2026  : {len(df_emi)}")
print(f"   - Registros externos  : {len(df_externo)}")
print(f"   - Total corpus        : {len(df_corpus)}")
print(f"   - Áreas cubiertas     : {df_corpus['area'].nunique()}")
print()

print("Distribución por fuente y área:")
print(df_corpus.groupby(['FUENTE','area']).size().unstack(fill_value=0).to_string())

## 💾 7. Exportación para notebooks siguientes

In [ ]:
# Exportar dataset limpio
df.to_csv('emi_contratos_limpio.csv', index=False, encoding='utf-8-sig')

# Exportar corpus unificado
df_corpus.to_csv('corpus_unificado.csv', index=False, encoding='utf-8-sig')

# Exportar resumen de áreas
resumen_areas = df.groupby('AREA').agg(
    contratos=('NRO','count'),
    docentes=('CEDULA','nunique'),
    asignaturas=('ASIGNATURA','nunique'),
    monto_total=('MONTO','sum'),
    monto_promedio=('MONTO','mean')
).round(2)
resumen_areas.to_csv('resumen_areas.csv', encoding='utf-8-sig')

print("✅ Archivos exportados:")
print("   - emi_contratos_limpio.csv")
print("   - corpus_unificado.csv")
print("   - resumen_areas.csv")
print()
print("Resumen por áreas:")
print(resumen_areas.to_string())

# Descargar archivos
from google.colab import files
files.download('emi_contratos_limpio.csv')
files.download('corpus_unificado.csv')
files.download('eda_contratos.png')

---
## ✅ Resumen del Notebook 1

| Tarea | Estado |
|---|---|
| Carga del dataset EMI 2026 (400 registros) | ✅ |
| Limpieza y normalización de campos | ✅ |
| Inferencia de áreas temáticas | ✅ |
| Análisis exploratorio (EDA) con gráficos | ✅ |
| Integración con corpus externo (Big Data) | ✅ |
| Exportación del corpus unificado | ✅ |

**Siguiente paso:** Notebook 2 — TF-IDF, Búsqueda con Ranking y Naive Bayes